# 📚 Multi-Agent Research Assistant — Kompletna Analiza Edukacyjna

**Autor notatki**: Analiza automatyczna  
**Cel**: Dogłębne zrozumienie architektury, technologii i wzorców projektowych  
**Poziom**: Średniozaawansowany → Zaawansowany

---

## 📋 Spis Treści

1. [Wprowadzenie — Czym jest projekt?](#1-wprowadzenie)
2. [Architektura Multi-Agent](#2-architektura-multi-agent)
3. [Stack Technologiczny](#3-stack-technologiczny)
4. [LangGraph — Orkiestracja Agentów](#4-langgraph)
5. [Ollama — Lokalne Modele LLM](#5-ollama)
6. [Struktura Kodu — Analiza Modułów](#6-struktura-kodu)
7. [State Management — Stan Współdzielony](#7-state-management)
8. [Wzorce Projektowe](#8-wzorce-projektowe)
9. [Konfiguracja z Pydantic Settings](#9-konfiguracja)
10. [Testowanie](#10-testowanie)
11. [Alternatywne Technologie](#11-alternatywy)
12. [Ćwiczenia Praktyczne](#12-cwiczenia)

---

# 1. Wprowadzenie — Czym jest projekt? {#1-wprowadzenie}

## 🎯 Cel projektu

**Multi-Agent Research Assistant** to system AI, który automatyzuje:
1. **Wyszukiwanie informacji w internecie** (web scraping)
2. **Weryfikację faktów** (fact-checking)
3. **Generowanie raportów** w formacie Markdown

## 🔑 Kluczowe cechy

| Cecha | Opis |
|-------|------|
| **Zero kosztów API za LLM** | Modele działają lokalnie przez Ollama |
| **Multi-agent** | 4 wyspecjalizowane agenty współpracujące |
| **Cykliczny workflow** | Supervisor może iterować dla lepszych wyników |
| **Weryfikacja faktów** | Automatyczne sprawdzanie twierdzeń |

## 💡 Dlaczego warto się tym zainteresować?

- **Praktyczne zastosowanie AI** — nie teoria, a działający produkt
- **Nowoczesny stack** — LangGraph, Ollama, Pydantic v2
- **Wzorzec multi-agent** — przyszłość systemów AI
- **Lokalne LLM** — prywatność danych, brak kosztów

---

# 2. Architektura Multi-Agent {#2-architektura-multi-agent}

## 🏗️ Diagram przepływu

```
User Query
    │
    ▼
┌─────────────────┐
│   SUPERVISOR    │ ← qwen2.5:7b-instruct (router/decydent)
│   (Orkiestrator) │
└────────┬────────┘
         │ (conditional edges — warunkowe krawędzie)
    ┌────┼──────────────────┐
    │    │                  │
    ▼    ▼                  ▼
┌────────────┐ ┌──────────────┐ ┌──────────────┐
│ RESEARCHER │ │   VERIFIER   │ │ SYNTHESIZER  │
│ mistral:7b │ │  mistral:7b  │ │  mistral:7b  │
│            │ │              │ │              │
│ Szuka info │ │ Sprawdza     │ │ Pisze        │
│ w sieci    │ │ fakty        │ │ raport       │
└────────────┘ └──────────────┘ └──────────────┘
    │                │                  │
    └────────────────┴──────────────────┘
                     │
                     ▼
              ┌─────────────┐
              │ FINAL REPORT│
              │  (Markdown) │
              └─────────────┘
```

## 🔄 Cykliczność grafu

Graf jest **cykliczny** — Supervisor może:
- Wrócić do Researchera jeśli potrzeba więcej danych
- Ponowić weryfikację
- Iterować do `max_iterations`

## 🤔 Dlaczego 4 agenty zamiast 1?

| Podejście | Zalety | Wady |
|-----------|--------|------|
| **1 agent** | Prostszy kod | Prompt explosion, trudne debugowanie |
| **Multi-agent** | Separacja odpowiedzialności, łatwiejsze testowanie, możliwość różnych modeli | Więcej kodu, komunikacja między agentami |

**Zasada Single Responsibility** — każdy agent ma jedną, jasno określoną funkcję.

---

# 3. Stack Technologiczny {#3-stack-technologiczny}

## 📦 Główne biblioteki

```python
# requirements.txt — analiza
langgraph>=0.2.0          # Orkiestracja agentów (graf stanów)
langchain>=0.3.0          # Podstawowe abstrakcje LLM
langchain-ollama>=0.2.0   # Integracja z Ollama
langchain-community>=0.3.0 # Narzędzia społeczności (np. Tavily)
tavily-python>=0.5.0      # API do wyszukiwania w internecie
python-dotenv>=1.0.0      # Ładowanie zmiennych z .env
pydantic>=2.0.0           # Walidacja danych
pydantic-settings>=2.0.0  # Konfiguracja aplikacji
rich>=13.0.0              # Kolorowy output w terminalu
```

## 🎯 Dlaczego te technologie?

### LangGraph vs LangChain Agents vs AutoGPT

| Technologia | Opis | Kiedy używać? |
|-------------|------|---------------|
| **LangGraph** ✅ | Graf stanów z pełną kontrolą przepływu | Złożone, deterministyczne workflow |
| **LangChain Agents** | Reaktywne agenty z tool-calling | Proste Q&A z narzędziami |
| **AutoGPT/CrewAI** | Autonomiczne agenty z minimalną kontrolą | Eksperymenty, nie produkcja |

**Wybrano LangGraph** ponieważ:
1. **Pełna kontrola** nad kolejnością wykonania
2. **Debugowalność** — wiadomo co się dzieje na każdym kroku
3. **Determinizm** — ten sam input = podobny output
4. **Type safety** — TypedDict dla stanu

### Ollama vs OpenAI API vs vLLM

| Rozwiązanie | Koszt | Prywatność | Setup |
|-------------|-------|------------|-------|
| **Ollama** ✅ | $0 | Pełna (lokalne) | Prosty |
| **OpenAI API** | $$/zapytanie | Dane w chmurze | Bardzo prosty |
| **vLLM** | $0 | Pełna | Złożony |
| **Hugging Face** | $0 | Pełna | Średni |

**Wybrano Ollama** ponieważ:
1. **Jeden plik binarny** — instalacja przez curl
2. **Automatyczna obsługa GPU** — wykrywa CUDA/ROCm
3. **API kompatybilne z OpenAI** — łatwa migracja
4. **Model hub** — `ollama pull` jak Docker

### Tavily vs Google Search API vs SerpAPI

| API | Darmowy tier | Jakość | Dla AI? |
|-----|--------------|--------|----------|
| **Tavily** ✅ | 1000 req/mies | Wysoka | Tak, dedykowane |
| **SerpAPI** | 100 req/mies | Bardzo wysoka | Nie |
| **Google Custom Search** | 100 req/dzień | Średnia | Nie |

**Wybrano Tavily** ponieważ:
1. **Zoptymalizowane dla AI** — zwraca czysty tekst, nie HTML
2. **Hojny darmowy tier** — 1000 zapytań/miesiąc
3. **Oficjalne wsparcie LangChain** — gotowy tool

---

# 4. LangGraph — Orkiestracja Agentów {#4-langgraph}

## 📖 Czym jest LangGraph?

**LangGraph** to biblioteka do budowania **stanowych, cyklicznych grafów** dla aplikacji LLM.

### Kluczowe koncepcje:

1. **StateGraph** — główna klasa definiująca graf
2. **Node** — funkcja przetwarzająca stan
3. **Edge** — połączenie między węzłami
4. **Conditional Edge** — routing warunkowy
5. **State** — współdzielony słownik danych

## 🔍 Analiza kodu: workflow.py

```python
# src/graph/workflow.py — linia po linii

from langgraph.graph import END, START, StateGraph
# END, START — specjalne węzły oznaczające początek i koniec
# StateGraph — klasa grafu stanów

def create_research_graph() -> Any:
    # 1. Tworzymy graf z typem stanu
    workflow = StateGraph(ResearchState)
    
    # 2. Dodajemy węzły (każdy to funkcja Python)
    workflow.add_node("supervisor", supervisor_node)
    workflow.add_node("researcher", researcher_node)
    workflow.add_node("verifier", verifier_node)
    workflow.add_node("synthesizer", synthesizer_node)
    
    # 3. Krawędź startowa (zawsze zaczyna od supervisora)
    workflow.add_edge(START, "supervisor")
    
    # 4. KLUCZOWE: Warunkowe krawędzie z supervisora
    workflow.add_conditional_edges(
        "supervisor",           # Węzeł źródłowy
        route_from_supervisor,  # Funkcja decyzyjna
        {                       # Mapowanie decyzji → węzeł
            "researcher": "researcher",
            "verifier": "verifier",
            "synthesizer": "synthesizer",
            "FINISH": END,
        },
    )
    
    # 5. Po każdym agencie wracamy do supervisora (cykl!)
    workflow.add_edge("researcher", "supervisor")
    workflow.add_edge("verifier", "supervisor")
    workflow.add_edge("synthesizer", "supervisor")
    
    # 6. Kompilacja — walidacja i optymalizacja
    return workflow.compile()
```

## 🎓 Funkcja routingu

```python
def route_from_supervisor(state: dict[str, Any]) -> str:
    """Odczytuje decyzję supervisora ze stanu."""
    next_agent: str = state.get("next_agent", "FINISH")
    
    # Walidacja — nieznane agenty = FINISH
    if next_agent not in {"researcher", "verifier", "synthesizer", "FINISH"}:
        return "FINISH"
    return next_agent
```

**Dlaczego osobna funkcja?**
- Separacja logiki decyzyjnej od definicji grafu
- Łatwe testowanie jednostkowe
- Możliwość dodania logowania/metryk

In [ ]:
# Uruchom ten blok aby zobaczyć jak wygląda graf
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.graph.workflow import create_research_graph

# Kompilacja grafu
graph = create_research_graph()

# Wyświetl strukturę grafu
print("Węzły w grafie:")
print(graph.get_graph().nodes)

---

# 5. Ollama — Lokalne Modele LLM {#5-ollama}

## 🖥️ Czym jest Ollama?

**Ollama** to lokalne środowisko uruchomieniowe dla modeli językowych.

```bash
# Instalacja (Linux)
curl -fsSL https://ollama.com/install.sh | sh

# Pobranie modelu
ollama pull qwen2.5:7b-instruct
ollama pull mistral:7b-instruct

# Uruchomienie serwera
ollama serve  # domyślnie http://localhost:11434
```

## 🧠 Modele użyte w projekcie

| Model | Rola | Parametry | VRAM |
|-------|------|-----------|------|
| **qwen2.5:7b-instruct** | Supervisor | 7B | ~5GB |
| **mistral:7b-instruct** | Researcher, Verifier, Synthesizer | 7B | ~5GB |

### Dlaczego różne modele?

```python
# src/config/settings.py
supervisor_model: str = "qwen2.5:7b-instruct"  # Lepszy w JSON
agent_model: str = "mistral:7b-instruct"       # Lepszy w generowaniu tekstu
```

**Qwen 2.5** — świetny w:
- Generowaniu strukturalnego JSON
- Podążaniu za instrukcjami
- Reasoning i planowanie

**Mistral 7B** — świetny w:
- Naturalnej generacji tekstu
- Podsumowywaniu
- Długich kontekstach

## 🔧 Integracja z LangChain

```python
from langchain_ollama import ChatOllama

# Tworzenie instancji LLM
llm = ChatOllama(
    model="qwen2.5:7b-instruct",
    base_url="http://localhost:11434",
    temperature=0.1,  # Niższa = bardziej deterministyczne
)

# Wywołanie
from langchain_core.messages import HumanMessage
response = llm.invoke([HumanMessage(content="Cześć!")])
print(response.content)
```

### Tryb JSON

```python
# Wymuszenie odpowiedzi JSON
llm_json = ChatOllama(
    model="mistral:7b-instruct",
    format="json",  # Kluczowy parametr!
)
```

## ⚠️ Alternatywy do Ollama

```python
# Alternatywa 1: OpenAI API
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", api_key="sk-...")

# Alternatywa 2: Anthropic Claude
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-3-haiku-20240307")

# Alternatywa 3: Lokalne przez Hugging Face
from langchain_huggingface import ChatHuggingFace
llm = ChatHuggingFace(model_id="mistralai/Mistral-7B-Instruct-v0.2")
```

---

# 6. Struktura Kodu — Analiza Modułów {#6-struktura-kodu}

## 📁 Organizacja projektu

```
multi-agent-research-assistant/
├── src/                          # Główny kod źródłowy
│   ├── __init__.py
│   ├── agents/                   # Funkcje węzłów grafu
│   │   ├── __init__.py
│   │   ├── supervisor.py         # 🧠 Orkiestrator
│   │   ├── researcher.py         # 🔍 Wyszukiwanie
│   │   ├── verifier.py           # ✅ Weryfikacja
│   │   └── synthesizer.py        # 📝 Raportowanie
│   ├── graph/                    # Definicja grafu LangGraph
│   │   ├── state.py              # TypedDict stanu
│   │   └── workflow.py           # Budowa grafu
│   ├── tools/                    # Zewnętrzne narzędzia
│   │   └── search.py             # Tavily wrapper
│   ├── prompts/                  # Szablony promptów
│   │   └── templates.py
│   └── config/                   # Konfiguracja
│       └── settings.py           # Pydantic settings
├── tests/                        # Testy jednostkowe
├── scripts/                      # Skrypty pomocnicze
├── notebooks/                    # Jupyter notebooks
├── reports/                      # Wygenerowane raporty
├── main.py                       # CLI entry point
├── requirements.txt
└── pyproject.toml
```

## 🎯 Dlaczego taka struktura?

**Podział na domeny:**
1. `agents/` — logika biznesowa agentów
2. `graph/` — infrastruktura orkiestracji
3. `tools/` — integracje zewnętrzne
4. `config/` — konfiguracja
5. `prompts/` — szablony tekstowe (łatwa modyfikacja bez zmian kodu)

**Korzyści:**
- **Testowalność** — można testować agenty w izolacji
- **Rozszerzalność** — łatwo dodać nowego agenta
- **Czytelność** — wiadomo gdzie szukać czego

---

# 7. State Management — Stan Współdzielony {#7-state-management}

## 📊 Definicja stanu

```python
# src/graph/state.py
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ResearchState(TypedDict):
    """Współdzielony stan przepływający przez graf."""
    
    # === Input użytkownika ===
    query: str                              # Pytanie badawcze
    
    # === Historia komunikacji ===
    messages: Annotated[list[BaseMessage], add_messages]
    # ⬆️ add_messages = automatyczne dołączanie (nie nadpisywanie)
    
    # === Dane z researchu ===
    search_results: list[dict]              # Wyniki Tavily
    research_notes: str                     # Notatki w Markdown
    key_claims: list[str]                   # Wyodrębnione twierdzenia
    
    # === Dane z weryfikacji ===
    verification_results: list[dict]        # [{claim, verdict, evidence}]
    verified_claims: list[str]              # Potwierdzone twierdzenia
    disputed_claims: list[str]              # Zakwestionowane twierdzenia
    
    # === Output ===
    final_report: str                       # Finalny raport Markdown
    
    # === Metadane orkiestracji ===
    current_agent: str                      # Kto właśnie działa
    next_agent: str                         # Decyzja supervisora
    iteration: int                          # Licznik pętli
    max_iterations: int                     # Limit pętli
    is_complete: bool                       # Czy zakończono
    errors: list[str]                       # Log błędów
```

## 🔑 Kluczowa koncepcja: Annotated + add_messages

```python
messages: Annotated[list[BaseMessage], add_messages]
```

**Co to robi?**
- `Annotated` — dodaje metadane do typu
- `add_messages` — **reducer** (funkcja agregująca)

**Jak działa reducer?**

```python
# Bez reducera (nadpisywanie):
state["data"] = ["nowa_wartość"]  # Stare dane znikają

# Z reducerem add_messages (dołączanie):
state["messages"] = [nowa_wiadomość]  # Dołącza do istniejących
```

**Dlaczego to ważne?**
- Każdy agent może dodawać wiadomości
- Historia jest zachowana
- Supervisor widzi całą konwersację

## 🏭 Factory function

```python
def create_initial_state(query: str, max_iterations: int = 3) -> ResearchState:
    """Tworzy czysty stan początkowy."""
    return ResearchState(
        query=query,
        messages=[],
        search_results=[],
        research_notes="",
        key_claims=[],
        verification_results=[],
        verified_claims=[],
        disputed_claims=[],
        final_report="",
        current_agent="supervisor",
        next_agent="",
        iteration=0,
        max_iterations=max_iterations,
        is_complete=False,
        errors=[],
    )
```

**Dlaczego factory function?**
- Gwarantuje wszystkie wymagane pola
- Sensowne wartości domyślne
- Łatwe tworzenie stanu w testach

In [ ]:
# Przykład tworzenia i inspekcji stanu
import sys
sys.path.insert(0, '..')

from src.graph.state import create_initial_state

# Tworzenie stanu
state = create_initial_state(
    query="Jakie są najnowsze trendy w AI?",
    max_iterations=5
)

# Wyświetlenie struktury
print("Struktura stanu:")
for key, value in state.items():
    print(f"  {key}: {type(value).__name__} = {repr(value)[:50]}...")

---

# 8. Wzorce Projektowe {#8-wzorce-projektowe}

## 🎨 Wzorce użyte w projekcie

### 1. **Supervisor Pattern** (Multi-Agent)

```
┌─────────────┐
│  SUPERVISOR │  ← Centralny decydent
└──────┬──────┘
       │
   ┌───┼───┐
   │   │   │
   ▼   ▼   ▼
  A1  A2  A3    ← Wyspecjalizowane agenty
```

**Alternatywa: Hierarchical (drzewo)**
```
      MANAGER
       /   \
   LEAD1   LEAD2
    / \     / \
   A1 A2  A3 A4
```

**Alternatywa: Peer-to-peer (sieć)**
```
   A1 ←→ A2
    \   /
     A3
```

**Wybrano Supervisor** ponieważ:
- Prostsze debugowanie
- Jasna odpowiedzialność
- Łatwa kontrola przepływu

---

### 2. **Factory Pattern** (Tworzenie LLM)

```python
# src/config/settings.py
class Settings(BaseSettings):
    def get_supervisor_llm(self) -> ChatOllama:
        """Factory method dla supervisora."""
        return ChatOllama(
            model=self.supervisor_model,
            base_url=self.ollama_base_url,
            temperature=self.model_temperature,
        )
    
    def get_agent_llm_json(self) -> ChatOllama:
        """Factory method z trybem JSON."""
        return ChatOllama(
            model=self.agent_model,
            format="json",  # Różnica!
            # ...
        )
```

**Korzyści:**
- Centralne zarządzanie konfiguracją
- Łatwa zmiana modelu w jednym miejscu
- Konsekwentne parametry

---

### 3. **Singleton Pattern** (Settings)

```python
from functools import lru_cache

@lru_cache  # ← Singleton przez cache'owanie
def get_settings() -> Settings:
    """Cached singleton for application settings."""
    return Settings()
```

**Dlaczego `lru_cache` a nie klasyczny Singleton?**
- Prostsze — brak boilerplate code
- Pythonowe — idiomatyczne rozwiązanie
- Możliwość `get_settings.cache_clear()` w testach

---

### 4. **Fallback Pattern** (Graceful Degradation)

```python
# src/agents/supervisor.py
try:
    response = llm.invoke([...])
    parsed = json.loads(response.content)
except Exception as exc:
    # Fallback do deterministycznej logiki
    fallback = _fallback_routing(state)
    next_agent = fallback["next_agent"]

def _fallback_routing(state: dict) -> dict:
    """Deterministyczny fallback."""
    if not state.get("research_notes"):
        return {"next_agent": "researcher"}
    if not state.get("verification_results"):
        return {"next_agent": "verifier"}
    # ...
```

**Dlaczego fallback?**
- LLM może zwrócić niepoprawny JSON
- Sieć może być niestabilna
- System nadal działa nawet przy błędach

---

# 9. Konfiguracja z Pydantic Settings {#9-konfiguracja}

## 📝 Pydantic Settings — Co to?

**pydantic-settings** to biblioteka do zarządzania konfiguracją z:
- Automatycznym ładowaniem z `.env`
- Walidacją typów
- Wartościami domyślnymi
- Type hints

## 🔍 Analiza kodu

```python
# src/config/settings.py
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    """Centralna konfiguracja aplikacji."""
    
    # Konfiguracja zachowania Pydantic
    model_config = SettingsConfigDict(
        env_file=".env",          # Plik z zmiennymi
        env_file_encoding="utf-8",
        case_sensitive=False,      # TAVILY_API_KEY = tavily_api_key
        extra="ignore",            # Ignoruj nieznane zmienne
    )
    
    # Zmienne z wartościami domyślnymi
    tavily_api_key: str = ""           # Wymagane (puste = błąd)
    ollama_base_url: str = "http://localhost:11434"
    supervisor_model: str = "qwen2.5:7b-instruct"
    agent_model: str = "mistral:7b-instruct"
    max_research_loops: int = 3
    max_search_results: int = 5
    model_temperature: float = 0.1
```

## 📄 Plik .env

```bash
# .env.example
TAVILY_API_KEY=tvly-your-key-here
OLLAMA_BASE_URL=http://localhost:11434
SUPERVISOR_MODEL=qwen2.5:7b-instruct
AGENT_MODEL=mistral:7b-instruct
```

## 🔄 Priorytet wartości

```
1. Zmienne środowiskowe (export TAVILY_API_KEY=...)
2. Plik .env
3. Wartości domyślne w klasie
```

## ⚠️ Alternatywy do Pydantic Settings

| Biblioteka | Zalety | Wady |
|------------|--------|------|
| **pydantic-settings** ✅ | Walidacja, type hints, IDE support | Zależność |
| **python-dotenv** | Prosta, minimalna | Brak walidacji |
| **dynaconf** | Wiele formatów (YAML, TOML) | Większa złożoność |
| **environs** | Prosta, walidacja | Mniej popularna |

In [ ]:
# Demonstracja ładowania konfiguracji
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.config.settings import get_settings

settings = get_settings()

print("=== Konfiguracja aplikacji ===")
print(f"Ollama URL:        {settings.ollama_base_url}")
print(f"Supervisor model:  {settings.supervisor_model}")
print(f"Agent model:       {settings.agent_model}")
print(f"Temperature:       {settings.model_temperature}")
print(f"Max iterations:    {settings.max_research_loops}")
print(f"Tavily API key:    {settings.tavily_api_key[:10]}...")

---

# 10. Testowanie {#10-testowanie}

## 🧪 Struktura testów

```
tests/
├── __init__.py
├── test_graph.py       # Testy grafu (nie wymagają Ollama)
├── test_agents.py      # Smoke testy agentów (wymagają Ollama)
└── test_tools.py       # Testy narzędzi (Tavily)
```

## 📖 Analiza test_graph.py

```python
# tests/test_graph.py
import pytest
from src.graph.state import ResearchState, create_initial_state

class TestResearchState:
    """Testy tworzenia stanu."""
    
    def test_create_initial_state_defaults(self) -> None:
        """Sprawdza wszystkie pola domyślne."""
        state = create_initial_state("test query")
        
        assert state["query"] == "test query"
        assert state["messages"] == []
        assert state["iteration"] == 0
        assert state["is_complete"] is False
        # ...

class TestRouteFromSupervisor:
    """Testy funkcji routingu."""
    
    def test_routes_to_researcher(self) -> None:
        state = {"next_agent": "researcher"}
        assert route_from_supervisor(state) == "researcher"
    
    def test_unknown_agent_routes_to_finish(self) -> None:
        """Nieznany agent = FINISH (graceful degradation)."""
        state = {"next_agent": "unknown_agent"}
        assert route_from_supervisor(state) == "FINISH"
```

## 🎯 Strategie testowania

| Poziom | Co testuje? | Zależności |
|--------|-------------|------------|
| **Unit** (state, routing) | Logika bez LLM | Brak |
| **Integration** (graph) | Kompilacja grafu | Brak |
| **Smoke** (agents) | Czy agent odpowiada | Ollama |
| **E2E** (full pipeline) | Cały przepływ | Ollama + Tavily |

## 🚫 Pomijanie testów zależnych

```python
# tests/test_agents.py
import os
import pytest

# Pomijaj całą klasę jeśli Ollama niedostępna
pytestmark = pytest.mark.skipif(
    os.getenv("SKIP_OLLAMA_TESTS", "true").lower() == "true",
    reason="Ollama tests are skipped by default.",
)

class TestSupervisorNode:
    def test_supervisor_returns_next_agent(self) -> None:
        # Ten test wymaga działającej Ollama
        ...
```

## ▶️ Uruchamianie testów

```bash
# Wszystkie testy (bez Ollama)
python -m pytest tests/ -v

# Tylko testy grafu
python -m pytest tests/test_graph.py -v

# Z testami Ollama
SKIP_OLLAMA_TESTS=false python -m pytest tests/ -v
```

---

# 11. Alternatywne Technologie {#11-alternatywy}

## 🔄 Dla każdego komponentu

### Agent Framework

| Alternatywa | Opis | Kiedy wybrać? |
|-------------|------|---------------|
| **LangGraph** ✅ | Graf stanów | Złożone, kontrolowane workflow |
| **AutoGen** | Konwersacje agentów | Research, brainstorming |
| **CrewAI** | Role-based agenty | Szybki prototyp |
| **LlamaIndex Workflows** | Pipelines danych | RAG-centric apps |

### LLM Provider

```python
# Przykłady zamienników:

# 1. OpenAI (cloud, płatne)
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. Google Gemini (cloud, darmowy tier)
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# 3. Groq (cloud, bardzo szybkie, darmowe limity)
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-70b-versatile")

# 4. Together AI (cloud, open-source models)
from langchain_together import ChatTogether
llm = ChatTogether(model="meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo")
```

### Search API

| API | Charakterystyka |
|-----|------------------|
| **Tavily** ✅ | AI-optimized, 1000 free/month |
| **SerpAPI** | Dokładne, drogie |
| **Brave Search** | Darmowe, prywatność |
| **DuckDuckGo** | Darmowe, bez API key |

```python
# Alternatywa: DuckDuckGo (darmowe, bez klucza)
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults()
results = search.invoke("quantum computing")
```

### CLI Framework

| Biblioteka | Zalety |
|------------|--------|
| **argparse** ✅ | Wbudowane, standard library |
| **typer** | Type hints, automatyczne help |
| **click** | Dekoratory, composable |
| **fire** | Auto-generate z klas/funkcji |

```python
# Alternatywa: Typer (bardziej nowoczesne)
import typer

app = typer.Typer()

@app.command()
def research(
    query: str = typer.Option(..., "--query", "-q"),
    verbose: bool = typer.Option(False, "--verbose", "-v"),
):
    """Run research pipeline."""
    ...

if __name__ == "__main__":
    app()
```

---

# 12. Ćwiczenia Praktyczne {#12-cwiczenia}

## 🎯 Ćwiczenie 1: Uruchom pipeline

**Cel:** Zrozumienie przepływu od początku do końca.

```bash
# Terminal
cd multi-agent-research-assistant
python main.py --query "Co to jest LangGraph?" --verbose
```

**Obserwuj:**
1. Jak Supervisor podejmuje decyzje
2. Jak Researcher wyciąga twierdzenia
3. Jak Verifier je weryfikuje
4. Jak wygląda finalny raport

---

## 🎯 Ćwiczenie 2: Dodaj nowego agenta

**Cel:** Rozszerzenie systemu o agenta `translator`.

**Kroki:**
1. Stwórz `src/agents/translator.py`
2. Dodaj węzeł w `workflow.py`
3. Zaktualizuj prompt supervisora
4. Rozszerz stan o pole `translated_report`

---

## 🎯 Ćwiczenie 3: Zmień model LLM

**Cel:** Przetestowanie różnych modeli.

```bash
# 1. Pobierz inny model
ollama pull llama3.2:3b

# 2. Zmień w .env
AGENT_MODEL=llama3.2:3b

# 3. Uruchom i porównaj wyniki
python main.py --query "Wyjaśnij computing kwantowy" --verbose
```

---

## 🎯 Ćwiczenie 4: Napisz test jednostkowy

**Cel:** Zrozumienie testowania agentów.

```python
# tests/test_my_feature.py
import pytest
from src.graph.state import create_initial_state
from src.agents.supervisor import _fallback_routing

class TestFallbackRouting:
    def test_empty_state_routes_to_researcher(self):
        """Pusty stan → researcher"""
        state = create_initial_state("test")
        result = _fallback_routing(state)
        assert result["next_agent"] == "researcher"
    
    def test_after_research_routes_to_verifier(self):
        """Po researchu → verifier"""
        state = create_initial_state("test")
        state["research_notes"] = "Some notes"
        result = _fallback_routing(state)
        assert result["next_agent"] == "verifier"
```

---

## 🎯 Ćwiczenie 5: Wizualizacja grafu

**Cel:** Zobaczenie struktury grafu.

In [ ]:
# Ćwiczenie 5: Wizualizacja grafu
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from src.graph.workflow import create_research_graph

graph = create_research_graph()

# Jeśli masz zainstalowany graphviz:
try:
    from IPython.display import Image, display
    # Graf w formacie Mermaid
    print(graph.get_graph().draw_mermaid())
except Exception as e:
    print(f"Nie można wyrenderować grafu: {e}")
    print("\nStruktura grafu:")
    print(graph.get_graph())

---

# 📚 Podsumowanie

## Czego się nauczyłeś?

1. **Architektura multi-agent** — Supervisor pattern z wyspecjalizowanymi agentami
2. **LangGraph** — Stanowe grafy dla orkiestracji LLM
3. **Ollama** — Lokalne uruchamianie modeli językowych
4. **State management** — TypedDict z reducerami
5. **Wzorce projektowe** — Factory, Singleton, Fallback
6. **Konfiguracja** — Pydantic Settings z walidacją
7. **Testowanie** — Strategie dla systemów AI

## 🚀 Co dalej?

1. **Dodaj pamięć** — Checkpointing w LangGraph
2. **Human-in-the-loop** — Interrupt przed weryfikacją
3. **Streaming** — Wyświetlanie odpowiedzi w czasie rzeczywistym
4. **RAG** — Dodaj bazę wiedzy (Chroma, Pinecone)
5. **LangSmith** — Włącz tracing dla debugowania

## 📖 Zasoby

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Ollama Docs](https://ollama.com/)
- [Pydantic Settings](https://docs.pydantic.dev/latest/concepts/pydantic_settings/)
- [Tavily API](https://docs.tavily.com/)

---

*Notatka wygenerowana automatycznie dla celów edukacyjnych.*